### Esercizi SQL Esteso

Sono date le seguenti tabelle:

    **Cliente**(*IdCliente*, Cliente, Provincia, Regione)

    **Categoria**(*IdCat*, Categoria)

    **Agente**(*IdAgente*, Agente, Agenzia)

    **Tempo**(*IdTempo*, Mese, Trimestre, Semestre, Anno)

    **Fatturato**(*IdTempo*, *IdCliente*, *IdCatArticolo*, *IdAgente*, TotFatturato, NumArticoli, TotSconto)

1. Visualizzare per ogni categoria di articoli
    • la categoria
    • la quantità totale fatturata per la categoria in esame
    • il fatturato totale associato alla categoria in esame
    • il rank della categoria in funzione della quantità totale fatturata
    • il rank della categoria in funzione del fatturato totale

In [ ]:
SELECT C.Categoria, SUM(F.NumArticoli) AS TotQuantità, SUM(F.TotFatturato) AS TotFatturato,
    RANK() OVER (
        ORDER BY SUM(F.NumArticoli) ASC
    ) AS PosClassificaQuantità,
    RANK() OVER (
        ORDER BY SUM(F.TotFatturato) ASC
    ) AS PasClassificaFatturato
FROM Fatturato F 
JOIN Categoria C ON C.IdCat = F.IdCat
GROUP BY C.Categoria;

2. Visualizzare per ogni provincia
    • la provincia
    • la regione della provincia
    • il fatturato totale associato alla provincia
    • il rank della provincia in funzione del fatturato totale, separato per regione

In [ ]:
SELECT C.Provincia, C.Regione, SUM(F.TotFatturato) AS FatturatoProvincia,
    RANK() OVER (
        PARTITION BY C.Regione
        ORDER BY SUM(F.TotFatturato) DESC
    ) AS PosClassificaProvinciaRegione
FROM Fatturato F 
JOIN Cliente C ON C.IdCliente = F.IdCliente
GROUP BY C.Provincia, C.Regione;

3. Visualizzare per ogni provincia e mese
    • la provincia
    • la regione della provincia
    • il mese
    • il fatturato totale associato alla provincia nel mese in esame
    • il rank della provincia in funzione del fatturato totale, separato per mese

In [ ]:
SELECT C.Provincia, C.Regione, T.Mese, SUM(F.TotFatturato) AS TotFatturatoProvinciaMese,
    RANK() OVER (
        PARTITION BY T.Mese
        ORDER BY SUM(F.TotFatturato) DESC
    ) AS PosClassificaProvinciaMese
FROM Fatturato F 
JOIN Cliente C ON C.IdCliente = F.IdCliente
JOIN Tempo T ON T.IdTempo = F.IdTempo
GROUP BY C.Provincia, C.Regione, T.Mese;

4. Visualizzare per ogni regione e mese
    • la regione
    • il mese
    • il fatturato totale associato alla regione nel mese in esame
    • l’incasso cumulativo al trascorrere dei mesi, separato per ogni regione
    • l’incasso cumulativo al trascorrere dei mesi, separato per ogni anno e regione

In [ ]:
SELECT C.Regione, T.Mese, SUM(F.TotFatturato) AS TotFatturatoRegioneMese,
    SUM(SUM(F.TotFatturato)) OVER (
        PARTITION BY C.Regione
        ORDER BY T.Mese ASC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS FatturatoCumulativoRegione,
    SUM(SUM(F.TotFatturato)) OVER (
        PARTITION BY T.Anno, C.Regione
        ORDER BY T.Mese ASC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS FatturatoCumulativoAnnoRegione,
FROM Fatturato F 
JOIN Cliente C ON C.IdCliente = F.IdCliente
JOIN Tempo T ON T.IdTempo = F.IdTempo
GROUP BY C.Regione, T.Mese, T.Anno;